In [0]:
from pyspark.sql.functions import current_timestamp, col

CAT, SCH = "capstone_five", "project"
LANDING = "/Volumes/capstone_five/project/landing"
CKPT    = "/Volumes/capstone_five/project/checkpoints"

query = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CKPT}/bronze_schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(f"{LANDING}/vitals/")
    .withColumn("_ingest_ts", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
  .writeStream
    .option("checkpointLocation", f"{CKPT}/bronze")
    .trigger(availableNow=True)
    .toTable(f"{CAT}.{SCH}.bronze_vitals"))

query.awaitTermination()
print("Rows in bronze_vitals:", spark.table(f"{CAT}.{SCH}.bronze_vitals").count())